# 🔄 Day 02a: LSP & ISP — Substitution & Segregation

---

## 🎯 What You'll Master Today
- LSP: Subtypes must be drop-in replacements for their base types
- ISP: Many small interfaces > one big fat interface
- Classic violations and how to fix them
- The Rectangle-Square problem (interview classic)

---

## 🎭 The Analogies

**LSP — Liskov Substitution Principle:**
- If you order a "vehicle" for your trip, any vehicle should work — car, van, bus
- But if someone sends you a **toy car**, it technically IS a vehicle, but it CAN'T do the job
- LSP says: subclasses must actually WORK wherever the parent is expected

**ISP — Interface Segregation Principle:**
- Don't force a **fish** to implement `fly()` just because it's an `Animal`
- Split into `Swimmable`, `Flyable`, `Walkable` — each animal picks what it actually can do

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  LSP & ISP — Core Ideas                                         ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  LSP: If S is a subtype of T, then objects of type T can be     ║
║       replaced with objects of type S WITHOUT breaking the       ║
║       program.                                                    ║
║                                                                   ║
║  ISP: No client should be forced to depend on methods it        ║
║       does not use.                                              ║
║                                                                   ║
║  LSP = about BEHAVIOR (subtypes must behave correctly)          ║
║  ISP = about INTERFACES (keep them small and focused)           ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. LSP VIOLATION — The Classic Rectangle-Square Problem
# ============================================

# ❌ BAD — Square IS-A Rectangle mathematically, but breaks LSP

class Rectangle:
    def __init__(self, width: float, height: float):
        self._width = width
        self._height = height
    
    @property
    def width(self):
        return self._width
    
    @width.setter
    def width(self, value):
        self._width = value
    
    @property
    def height(self):
        return self._height
    
    @height.setter
    def height(self, value):
        self._height = value
    
    def area(self):
        return self._width * self._height


class Square(Rectangle):  # ❌ Square IS-A Rectangle?
    def __init__(self, side: float):
        super().__init__(side, side)
    
    @Rectangle.width.setter
    def width(self, value):
        self._width = value
        self._height = value  # must keep both equal!
    
    @Rectangle.height.setter
    def height(self, value):
        self._width = value
        self._height = value


def test_area(rect: Rectangle):
    """This function expects Rectangle behavior."""
    rect.width = 5
    rect.height = 4
    expected = 20  # 5 × 4 = 20
    actual = rect.area()
    print(f"Expected: {expected}, Got: {actual}, {'✅' if actual == expected else '❌ LSP VIOLATED!'}")


print("Testing Rectangle:")
test_area(Rectangle(0, 0))  # ✅ Works: 20

print("\nTesting Square (substitute for Rectangle):")
test_area(Square(0))  # ❌ Fails: got 16 (4×4), not 20!

In [ ]:
# ============================================
# 2. LSP FIXED — Use composition, not inheritance
# ============================================
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        pass

class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height
    
    def area(self) -> float:
        return self.width * self.height

class Square(Shape):  # NOT inheriting from Rectangle!
    def __init__(self, side: float):
        self.side = side
    
    def area(self) -> float:
        return self.side ** 2


# Both are Shapes, but neither pretends to be the other
shapes = [Rectangle(5, 4), Square(4)]
for s in shapes:
    print(f"{s.__class__.__name__}: area = {s.area()}")  # ✅ Both work correctly

print("\n✅ LSP satisfied: each shape correctly implements area()")

In [ ]:
# ============================================
# 3. Another LSP Violation — Penguins Can't Fly
# ============================================

# ❌ BAD — Penguin IS-A Bird, but can't fly!
class Bird:
    def fly(self):
        return "Flying high! 🦅"

class Penguin(Bird):  # ❌ LSP violation
    def fly(self):
        raise NotImplementedError("Penguins can't fly! 🐧")


def make_bird_fly(bird: Bird):
    print(bird.fly())

make_bird_fly(Bird())  # ✅ Works
try:
    make_bird_fly(Penguin())  # ❌ Crashes!
except NotImplementedError as e:
    print(f"LSP Violated: {e}")

print("\n--- LSP Rule: If it raises NotImplementedError, it shouldn't inherit that method ---")

In [ ]:
# ============================================
# 4. ISP VIOLATION — Fat Interface
# ============================================
from abc import ABC, abstractmethod

# ❌ BAD — One FAT interface forces all animals to implement everything
class Animal(ABC):
    @abstractmethod
    def walk(self): pass
    
    @abstractmethod
    def swim(self): pass
    
    @abstractmethod
    def fly(self): pass


class Dog(Animal):
    def walk(self): return "Dog walking 🐕"
    def swim(self): return "Dog swimming 🐕"
    def fly(self):  raise NotImplementedError("Dogs can't fly!")  # ❌ Forced to implement!


print("❌ ISP Violation: Dog is forced to implement fly()")
dog = Dog()
print(dog.walk())
try:
    dog.fly()
except NotImplementedError as e:
    print(f"   {e}")

In [ ]:
# ============================================
# 5. ISP FIXED — Small, Focused Interfaces
# ============================================
from abc import ABC, abstractmethod

# ✅ GOOD — Split into focused interfaces
class Walkable(ABC):
    @abstractmethod
    def walk(self) -> str: pass

class Swimmable(ABC):
    @abstractmethod
    def swim(self) -> str: pass

class Flyable(ABC):
    @abstractmethod
    def fly(self) -> str: pass


# Each animal implements ONLY what it can do
class Dog(Walkable, Swimmable):  # No Flyable!
    def walk(self) -> str:
        return "Dog walking 🐕"
    def swim(self) -> str:
        return "Dog swimming 🐕"

class Eagle(Walkable, Flyable):  # No Swimmable!
    def walk(self) -> str:
        return "Eagle walking 🦅"
    def fly(self) -> str:
        return "Eagle soaring 🦅"

class Duck(Walkable, Swimmable, Flyable):  # All three!
    def walk(self) -> str:
        return "Duck waddling 🦆"
    def swim(self) -> str:
        return "Duck paddling 🦆"
    def fly(self) -> str:
        return "Duck flying 🦆"


# Type-safe usage
def make_it_fly(flyer: Flyable):
    print(flyer.fly())

def make_it_swim(swimmer: Swimmable):
    print(swimmer.swim())


make_it_fly(Eagle())  # ✅
make_it_fly(Duck())   # ✅
# make_it_fly(Dog())  # ❌ Type error — Dog isn't Flyable

make_it_swim(Dog())   # ✅
make_it_swim(Duck())  # ✅

print("\n✅ ISP: No animal is forced to implement methods it can't use")

In [ ]:
# ============================================
# 6. Real-World ISP — Worker Interfaces
# ============================================
from abc import ABC, abstractmethod

# ❌ BAD — Fat interface
# class Worker(ABC):
#     @abstractmethod
#     def work(self): pass
#     @abstractmethod
#     def eat(self): pass    # Robots don't eat!
#     @abstractmethod
#     def sleep(self): pass  # Robots don't sleep!

# ✅ GOOD — Segregated interfaces
class Workable(ABC):
    @abstractmethod
    def work(self) -> str: pass

class Eatable(ABC):
    @abstractmethod
    def eat(self) -> str: pass

class Sleepable(ABC):
    @abstractmethod
    def sleep(self) -> str: pass


class HumanWorker(Workable, Eatable, Sleepable):
    def work(self) -> str:
        return "Human working 💻"
    def eat(self) -> str:
        return "Human eating 🍕"
    def sleep(self) -> str:
        return "Human sleeping 😴"

class RobotWorker(Workable):  # Only Workable — no eat or sleep!
    def work(self) -> str:
        return "Robot working 24/7 🤖"


human = HumanWorker()
robot = RobotWorker()

print(human.work())
print(human.eat())
print(robot.work())
# robot.eat()  # ❌ Doesn't exist — and shouldn't!

## 🗺️ LSP & ISP Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  LSP & ISP — QUICK REFERENCE                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  LSP: Subtype must work ANYWHERE parent type is expected.       ║
║  Violation signs:                                                ║
║    • NotImplementedError in overridden methods                  ║
║    • Subclass changes preconditions/postconditions              ║
║    • Rectangle-Square problem                                    ║
║  Fix: Don't inherit if behavior changes. Use composition/ABC.   ║
║                                                                   ║
║  ISP: Split fat interfaces into focused ones.                   ║
║  Violation signs:                                                ║
║    • Classes forced to implement methods they don't use         ║
║    • Empty/stub implementations of abstract methods             ║
║  Fix: Many small ABCs. Classes implement only what they need.   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions — Practice These

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is LSP? | Objects of a superclass should be replaceable with objects of a subclass without breaking the program |
| 2 | Rectangle-Square problem? | Square IS-A Rectangle mathematically, but not behaviorally. Setting width shouldn't change height |
| 3 | How to detect LSP violation? | Subclass throws NotImplementedError, changes behavior, or breaks parent's contract |
| 4 | What is ISP? | No client should be forced to depend on methods it doesn't use |
| 5 | How to fix fat interfaces? | Split into small, focused interfaces. Classes implement only what they need |
| 6 | Robot worker example? | Robot shouldn't implement eat()/sleep(). Split Worker into Workable + Eatable + Sleepable |
| 7 | LSP vs OCP? | LSP = subtypes behavioral correctness. OCP = extend without modification. Both use abstraction |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • LSP: subtypes must be drop-in replacements            │
## │  • NotImplementedError in override = LSP violation        │
## │  • Rectangle-Square: don't inherit if behavior changes   │
## │  • ISP: many small interfaces > one fat interface        │
## │  • Split Flyable/Swimmable/Walkable, not one Animal ABC  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Dependency Inversion Principle** — depend on abstractions, not concretions